# 230968272
# Vetsa Anuraga Chandan
# Week 6

**Exercise 1**
Hill-Climbing Search for Feature Selection in Predictive Modeling
Problem Statement: You are given a dataset (select a random dataset) with multiple input
features and a target variable. Each state represents a subset of selected features. The goal is
to find a feature subset that maximizes model accuracy.

**State Representation**

1. A binary vector indicating whether a feature is selected (1) or not (0)

**Initial State**
1. A randomly generated feature subset


**Neighbor Generation**

1. Flip one bit (add/remove one feature)

**Evaluation Function**

1. Cross-validation accuracy of a simple classifier (e.g., Logistic Regression)

**Algorithm Implementation**

1. Implement steepest-ascent hill climbing

2. Stop when no neighbor improves the current solution

**Experimentation**

1. Run the algorithm multiple times with different random initial states

In [1]:
import numpy as np
import random
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [2]:
data = load_breast_cancer()
X= data.data
y=data.target

In [3]:
n_feature = X.shape[1]

In [4]:
def evaluate(state):
    if np.sum(state) == 0:
        return 0  
    
    selected_indices = np.where(state == 1)[0]
    X_subset = X[:, selected_indices]
    
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=5000))
    
    scores = cross_val_score(model, X_subset, y, cv=5)
    return scores.mean()

In [5]:
def generate_neighbors(state):
    neighbors = []
    for i in range(len(state)):
        neighbor = state.copy()
        neighbor[i] = 1 - neighbor[i] 
        neighbors.append(neighbor)
    return neighbors

In [6]:
def hill_climbing():
    current_state = np.random.randint(0, 2, n_feature)
    current_score = evaluate(current_state)
    
    while True:
        neighbors = generate_neighbors(current_state)
        scores = [evaluate(neighbor) for neighbor in neighbors]
        
        best_score = max(scores)
        best_neighbor = neighbors[np.argmax(scores)]
        
        if best_score <= current_score:
            break
        
        current_state = best_neighbor
        current_score = best_score
    
    return current_state, current_score

In [7]:
runs = 5
results = []

for i in range(runs):
    best_state, best_score = hill_climbing()
    selected_features = np.sum(best_state)
    results.append(best_score)
    
    print(f"Run {i+1}")
    print(f"Selected Features: {selected_features}")
    print(f"Best Accuracy: {best_score:.4f}")
    print("-" * 40)

print("Average Accuracy over runs:", np.mean(results))

Run 1
Selected Features: 15
Best Accuracy: 0.9807
----------------------------------------
Run 2
Selected Features: 17
Best Accuracy: 0.9772
----------------------------------------
Run 3
Selected Features: 18
Best Accuracy: 0.9789
----------------------------------------
Run 4
Selected Features: 16
Best Accuracy: 0.9842
----------------------------------------
Run 5
Selected Features: 17
Best Accuracy: 0.9807
----------------------------------------
Average Accuracy over runs: 0.9803322465455675


**Exercise 2**: Simulated Annealing for Vehicle Routing Optimization
Problem Statement: A delivery vehicle must visit a set of locations and return to the depot.
The order of visiting locations affects the total travel cost. Each state represents a permutation
of delivery locations.

**State Representation**
1. A sequence representing the visiting order of delivery points
   
**Initial State**
1. A random permutation of locations

**Neighbor Generation**
1. Swap two locations in the route
   
**Cost Function**
1. Total travel distance of the route
   
**Simulated Annealing Implementation** 
1. Implement SA using:
* Temperature schedule T(t)
* Acceptance probability:
−Δ E/ T
P=e


In [8]:
import numpy as np
import random
import math

np.random.seed(42)
random.seed(42)

n_locations = 15
locations = np.random.rand(n_locations, 2) * 100
depot = np.array([50, 50])

In [9]:
def total_distance(route):
    distance = 0
    current = depot
    for idx in route:
        next_loc = locations[idx]
        distance += np.linalg.norm(current - next_loc)
        current = next_loc
    distance += np.linalg.norm(current - depot)
    return distance

In [10]:
def random_route():
    route = list(range(n_locations))
    random.shuffle(route)
    return route

In [11]:
def neighbor(route):
    new_route = route.copy()
    i, j = random.sample(range(n_locations), 2)
    new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route

In [12]:
def simulated_annealing(T0=1000, alpha=0.995, Tmin=1e-3, max_iter=10000):
    current = random_route()
    current_cost = total_distance(current)
    best = current.copy()
    best_cost = current_cost
    T = T0
    iteration = 0
    while T > Tmin and iteration < max_iter:
        candidate = neighbor(current)
        candidate_cost = total_distance(candidate)
        delta = candidate_cost - current_cost
        if delta < 0 or random.random() < math.exp(-delta / T):
            current = candidate
            current_cost = candidate_cost
            if current_cost < best_cost:
                best = current.copy()
                best_cost = current_cost
        T *= alpha
        iteration += 1
    return best, best_cost

best_route, best_cost = simulated_annealing()

print("Best Route:", best_route)
print("Best Cost:", best_cost)



Best Route: [1, 6, 14, 10, 4, 12, 0, 5, 3, 13, 11, 7, 2, 9, 8]
Best Cost: 399.6160086161764


**Exercise 3 (Revised)**: Genetic Algorithm for Autonomous Drone Path Planning

**Problem Statement**: A drone must travel from a start location to a destination in a 2D
environment with obstacles. Each candidate solution represents a sequence of waypoints
defining a flight path. The goal is to evolve paths that minimize travel distance while
avoiding obstacles.
**Chromosome Representation**
1. A fixed-length sequence of (x, y) waypoints

**Initial Population**
1. Randomly generated feasible paths
2. Fitness Function

**Minimize:**

1. Total path length

2. Obstacle collisions (high penalty)

**Genetic Operators**

1. Selection: Tournament selection

2. Crossover: One-point crossover on waypoint sequence

3. Mutation: Random perturbation of waypoint coordinates

**Constraints**

1. Waypoints must remain within map boundaries

2. Paths intersecting obstacles are penalized

**Termination Condition**
1. Fixed number of generations or convergence

In [13]:
random.seed(42)
np.random.seed(42)

WIDTH = 100
HEIGHT = 100
START = (5, 5)
GOAL = (95, 95)

OBSTACLES = [
    (30, 30, 15),
    (70, 40, 12),
    (50, 75, 10)
]

POP_SIZE = 60
WAYPOINTS = 8
GENERATIONS = 150
TOURNAMENT_K = 3
MUTATION_RATE = 0.2
MUTATION_SCALE = 10
PENALTY = 10000

In [14]:
def distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

In [15]:
def segment_circle_collision(p1, p2, circle):
    cx, cy, r = circle
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    fx = p1[0] - cx
    fy = p1[1] - cy
    a = dx * dx + dy * dy
    b = 2 * (fx * dx + fy * dy)
    c = fx * fx + fy * fy - r * r
    discriminant = b * b - 4 * a * c
    if discriminant < 0:
        return False
    discriminant = math.sqrt(discriminant)
    t1 = (-b - discriminant) / (2 * a)
    t2 = (-b + discriminant) / (2 * a)
    return (0 <= t1 <= 1) or (0 <= t2 <= 1)


In [16]:
def path_collision(path):
    points = [START] + path + [GOAL]
    for i in range(len(points) - 1):
        for obs in OBSTACLES:
            if segment_circle_collision(points[i], points[i + 1], obs):
                return True
    return False


In [17]:
def path_length(path):
    points = [START] + path + [GOAL]
    return sum(distance(points[i], points[i + 1]) for i in range(len(points) - 1))


In [18]:
def fitness(path):
    length = path_length(path)
    if path_collision(path):
        return length + PENALTY
    return length


In [19]:
def random_waypoint():
    return (random.uniform(0, WIDTH), random.uniform(0, HEIGHT))


In [20]:
def create_individual():
    return [random_waypoint() for _ in range(WAYPOINTS)]

def tournament_selection(population):
    selected = random.sample(population, TOURNAMENT_K)
    selected.sort(key=lambda ind: fitness(ind))
    return selected[0]

In [21]:
def crossover(parent1, parent2):
    point = random.randint(1, WAYPOINTS - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

In [22]:
def mutate(individual):
    new_ind = []
    for x, y in individual:
        if random.random() < MUTATION_RATE:
            x += random.uniform(-MUTATION_SCALE, MUTATION_SCALE)
            y += random.uniform(-MUTATION_SCALE, MUTATION_SCALE)
        x = min(max(x, 0), WIDTH)
        y = min(max(y, 0), HEIGHT)
        new_ind.append((x, y))
    return new_ind

population = [create_individual() for _ in range(POP_SIZE)]

for _ in range(GENERATIONS):
    new_population = []
    while len(new_population) < POP_SIZE:
        parent1 = tournament_selection(population)
        parent2 = tournament_selection(population)
        child1, child2 = crossover(parent1, parent2)
        child1 = mutate(child1)
        child2 = mutate(child2)
        new_population.append(child1)
        new_population.append(child2)
    population = new_population[:POP_SIZE]

best = min(population, key=lambda ind: fitness(ind))
print(f"Best Path:{best}\n")
print("Best Fitness:", fitness(best))

Best Path:[(16.045776656324716, 4.512476716986255), (43.12691929161494, 15.448922690126834), (54.96184173978315, 32.06836337485186), (51.960827743482305, 40.75813413558251), (59.016831192579396, 54.56427121545299), (81.99189990048494, 76.7488382324805), (81.3723874116906, 76.52590924985095), (93.38897509550753, 92.4911351355847)]

Best Fitness: 140.9231438312816
